# 블로그·flat PKL 전처리 (`blog_flat_preprocess`)

`data/integrated/combined_section_sorted_flat_comments.pkl` 기준으로 **TF-IDF·KMeans·LDA 이전**까지 전처리합니다.
이후 분석은 [`03_analysis/section_analysis_pipeline.ipynb`](../03_analysis/section_analysis_pipeline.ipynb)를 실행하세요.


#### 시작 전 세팅

In [ ]:
# 기본 라이브러리
import ast
import sys
import itertools
from collections import Counter
from pathlib import Path
from matplotlib import font_manager

# 데이터 처리
import numpy as np
import pandas as pd

# 시각화
import matplotlib.pyplot as plt
from IPython.display import display
from wordcloud import WordCloud

# 분석 모델
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation

In [ ]:
# 현재 작업 폴더가 notebooks/ 하위 어디에 있든
# project_paths.py 를 기준으로 루트를 자동으로 찾습니다.
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "code" / "notebook_bootstrap.py").is_file():
        sys.path.insert(0, str(_d / "code"))
        break
else:
    raise FileNotFoundError("notebooks/lib/notebook_bootstrap.py 을 찾을 수 없습니다.")

from notebook_bootstrap import setup_paths

PROJECT_ROOT = setup_paths()
root = PROJECT_ROOT
print("프로젝트 루트:", root)


In [ ]:
# 프로젝트에서 공통으로 쓰는 경로 상수를 불러옵니다.
from project_paths import (
    CONFIG_STOPWORDS,
    DATA_INTEGRATED,
    OUTPUTS_PIPELINE,
    OUTPUTS_PIPELINE_KMEANS,
    OUTPUTS_PIPELINE_LDA,
    OUTPUTS_PIPELINE_TFIDF,
    OUTPUTS_PIPELINE_WORDCLOUD_FILTERED,
    OUTPUTS_PIPELINE_WORDCLOUD_RAW,
    ensure_output_dirs,
)

# 결과 저장 폴더가 없으면 만들어 둡니다.
ensure_output_dirs()

print("통합 데이터 폴더:", DATA_INTEGRATED)
print("불용어 폴더:", CONFIG_STOPWORDS)
print("결과 저장 폴더:", OUTPUTS_PIPELINE)

In [ ]:
# 경로 설정

# 분석 시작 파일
# 이 파일은 명사 컬럼이 이미 리스트 형태라서 바로 분석에 쓰기 좋습니다.
INPUT_FILE = DATA_INTEGRATED / "combined_section_sorted_flat_comments.pkl"

# 불용어 파일
# common은 전체 공통 불용어, local_section은 각 구간 전용 불용어입니다.
COMMON_STOPWORDS_FILE = CONFIG_STOPWORDS / "stopwords_common.txt"
SECTION_STOPWORDS_FILES = {
    1: CONFIG_STOPWORDS / "stopwords_local_section1.txt",
    2: CONFIG_STOPWORDS / "stopwords_local_section2.txt",
    3: CONFIG_STOPWORDS / "stopwords_local_section3.txt",
    4: CONFIG_STOPWORDS / "stopwords_local_section4.txt",
}

# 제목 명사까지 포함할지 여부
# False 면 본문 + 댓글만 사용합니다.
INCLUDE_TITLE_NOUNS = False

# 워드클라우드 한글 폰트 경로 (셀 "프로젝트 루트"에서 설정한 PROJECT_ROOT 사용)
FONT_PATH = str((PROJECT_ROOT / "nanum-gothic" / "NanumGothic.ttf").resolve())

font_manager.fontManager.addfont(FONT_PATH)
plt.rcParams["font.family"] = font_manager.FontProperties(fname=FONT_PATH).get_name()
plt.rcParams["axes.unicode_minus"] = False

# TF-IDF 전체 상위 단어 개수
TFIDF_TOP_N = 30

# K-means 클러스터 개수
KMEANS_N_CLUSTERS = 5

# LDA 토픽 개수
LDA_N_TOPICS = 5

# LDA 에서 토픽별 상위 단어 개수
LDA_TOP_WORDS = 15

In [ ]:
# 문자열로 저장된 리스트를 실제 리스트로 바꾸는 함수입니다.
# 이미 리스트면 그대로 반환합니다.
def ensure_token_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []

# txt 파일에서 불용어를 한 줄씩 읽는 함수입니다.
def load_stopwords(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"불용어 파일이 없습니다: {path}")

    return [
        line.strip()
        for line in path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip()
    ]


In [ ]:
# 워드클라우드를 화면에 보여주고, 원하면 png 파일로 저장하는 함수입니다.
def make_wordcloud(counter, title, output_file=None):
    wc = WordCloud(
        font_path=FONT_PATH,
        background_color="white",
        width=1400,
        height=900
    ).generate_from_frequencies(dict(counter))

    plt.figure(figsize=(12, 8))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.show()

    if output_file is not None:
        wc.to_file(str(output_file))

    return wc


# 4구간 라벨
SECTION_LABELS = {
    1: "1구간 (2024-01-01 ~ 2024-03-31)",
    2: "2구간 (2024-04-01 ~ 2024-06-30)",
    3: "3구간 (2024-07-01 ~ 2024-12-31)",
    4: "4구간 (2025-01-01 ~ 2025-06-30)"
}


# section 단위로 명사를 하나의 문서처럼 묶는 함수입니다.
# token_col 에 full_nouns_raw 또는 full_nouns_filtered 를 넣어서 사용합니다.
def build_section_df(source_df, token_col):
    section_df = (
        source_df.sort_values(["section", "date"])
        .groupby("section")
        .agg(
            period_start=("date", "min"),
            period_end=("date", "max"),
            post_count=(token_col, "size"),
            all_nouns=(token_col, lambda rows: list(itertools.chain.from_iterable(rows)))
        )
        .reset_index()
    )

    section_df["section_doc"] = section_df["all_nouns"].apply(lambda tokens: " ".join(tokens))
    section_df["section_label"] = section_df["section"].apply(
        lambda x: SECTION_LABELS.get(int(x), f"{x}구간")
    )

    return section_df

### 1. 전처리된 파일 불러오기

In [ ]:
# 분석에 사용할 전처리 완료 파일을 불러옵니다.
print("불러올 파일:", INPUT_FILE)
print("파일 존재 여부:", INPUT_FILE.exists())

df = pd.read_pickle(INPUT_FILE).copy()

# 날짜 컬럼은 시계열 분석과 section 정렬에 쓰기 때문에 datetime 으로 바꿉니다.
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("데이터 크기:", df.shape)
display(df.head(5))

In [ ]:
# 명사 컬럼이 혹시 문자열 상태여도 안전하게 리스트로 맞춰줍니다.
noun_cols = ["title_token_noun", "document_token_noun", "comment_token_noun"]

for col in noun_cols:
    df[col] = df[col].apply(ensure_token_list)

# 확인용 출력
for col in noun_cols:
    print(col, type(df[col].iloc[0]), df[col].iloc[0][:10])

### 2. 원본 명사 만들기

In [ ]:
# 기본 분석용 명사 컬럼을 만듭니다.
# 기본값은 본문 + 댓글 명사입니다.
# 제목도 포함하고 싶으면 INCLUDE_TITLE_NOUNS = True 로 바꾸면 됩니다.
if INCLUDE_TITLE_NOUNS:
    df["full_nouns_raw"] = (
        df["title_token_noun"]
        + df["document_token_noun"]
        + df["comment_token_noun"]
    )
else:
    df["full_nouns_raw"] = (
        df["document_token_noun"]
        + df["comment_token_noun"]
    )

# 벡터화에 쓰기 쉽게 공백으로 이어붙인 문자열 문서도 같이 만듭니다.
df["doc_text_raw"] = df["full_nouns_raw"].apply(lambda tokens: " ".join(tokens))

display(df[["title", "section", "full_nouns_raw"]].head())

### 3. 불용어 처리 전 워드 클라우드

In [ ]:
# 불용어 처리 전 워드클라우드는 전체 1장이 아니라 section별 4개로 만듭니다.
section_raw_df = build_section_df(df, "full_nouns_raw")

display(
    section_raw_df[["section", "section_label", "period_start", "period_end", "post_count"]]
)

In [ ]:
# 각 구간별 불용어 처리 전 상위 단어와 워드클라우드를 생성합니다.
for _, row in section_raw_df.iterrows():
    sec = int(row["section"])
    label = row["section_label"]

    counter = Counter(row["all_nouns"])

    top_df = pd.DataFrame(
        counter.most_common(100),
        columns=["keyword", "count"]
    )

    print(f"\n===== {label} 불용어 처리 전 상위 단어 =====")
    display(top_df.head(20))

    make_wordcloud(
        counter,
        title=f"{label} 불용어 처리 전 워드클라우드",
        output_file=OUTPUTS_PIPELINE_WORDCLOUD_RAW / f"wordcloud_raw_section_{sec}.png"
    )

    top_df.to_csv(
        OUTPUTS_PIPELINE_WORDCLOUD_RAW / f"wordcloud_raw_top100_section_{sec}.csv",
        index=False,
        encoding="utf-8-sig"
    )

### 4. 불용어 파일 불러오기

In [ ]:
print("공통 불용어 파일:", COMMON_STOPWORDS_FILE)
print("공통 파일 존재 여부:", COMMON_STOPWORDS_FILE.exists())

for sec, path in SECTION_STOPWORDS_FILES.items():
    print(f"{sec}구간 불용어 파일:", path)
    print("  파일 존재 여부:", path.exists())

common_stopwords = set(load_stopwords(COMMON_STOPWORDS_FILE))

section_local_stopwords_map = {
    sec: set(load_stopwords(path))
    for sec, path in SECTION_STOPWORDS_FILES.items()
}

section_stopwords_map = {
    sec: common_stopwords | local_stopwords
    for sec, local_stopwords in section_local_stopwords_map.items()
}

print("공통 불용어 개수:", len(common_stopwords))
for sec in sorted(section_stopwords_map):
    print(
        f"{sec}구간 불용어 개수: "
        f"local={len(section_local_stopwords_map[sec])}개, "
        f"common+local={len(section_stopwords_map[sec])}개"
    )

### 5. 불용어 적용

In [ ]:
def get_section_stopwords(section):
    if pd.isna(section):
        return common_stopwords
    return section_stopwords_map.get(int(section), common_stopwords)

# 1) 전체 집계용: common만 적용
df["full_nouns_common_filtered"] = df["full_nouns_raw"].apply(
    lambda tokens: [token for token in tokens if token not in common_stopwords]
)

# 2) 구간 분석용: common + 해당 구간 local 적용
df["full_nouns_filtered"] = df.apply(
    lambda row: [
        token
        for token in row["full_nouns_raw"]
        if token not in get_section_stopwords(row["section"])
    ],
    axis=1,
)

# 전체용 문자열 문서
df["doc_text_common_filtered"] = df["full_nouns_common_filtered"].apply(
    lambda tokens: " ".join(tokens)
)

# 구간용 문자열 문서
df["doc_text_filtered"] = df["full_nouns_filtered"].apply(
    lambda tokens: " ".join(tokens)
)

display(df[["title", "section", "full_nouns_common_filtered", "full_nouns_filtered"]].head())

In [ ]:
# 불용어 처리 후 명사를 다시 펼쳐서 빈도를 계산합니다.
filtered_tokens = list(itertools.chain.from_iterable(df["full_nouns_filtered"]))
filtered_counter = Counter(filtered_tokens)

filtered_top100 = pd.DataFrame(
    filtered_counter.most_common(100),
    columns=["keyword", "count"]
)

display(filtered_top100.head(20))

### 6. 불용어 처리 후 워드클라우드

In [ ]:
# 불용어 처리 후 워드클라우드도 section별 4개로 만듭니다.
section_filtered_df = build_section_df(df, "full_nouns_filtered")

display(
    section_filtered_df[["section", "section_label", "period_start", "period_end", "post_count"]]
)

In [ ]:
# 각 구간별 불용어 처리 후 상위 단어와 워드클라우드를 생성합니다.
for _, row in section_filtered_df.iterrows():
    sec = int(row["section"])
    label = row["section_label"]

    counter = Counter(row["all_nouns"])

    top_df = pd.DataFrame(
        counter.most_common(100),
        columns=["keyword", "count"]
    )

    print(f"\n===== {label} 불용어 처리 후 상위 단어 =====")
    display(top_df.head(20))

    make_wordcloud(
        counter,
        title=f"{label} 불용어 처리 후 워드클라우드",
        output_file=OUTPUTS_PIPELINE_WORDCLOUD_FILTERED / f"wordcloud_filtered_section_{sec}.png"
    )

    top_df.to_csv(
        OUTPUTS_PIPELINE_WORDCLOUD_FILTERED / f"wordcloud_filtered_top100_section_{sec}.csv",
        index=False,
        encoding="utf-8-sig"
    )